In [ ]:
from Datasets.mapping import nationality_to_country
from Datasets.utilities import country_name_standardizer
from Datasets.wiki_nationality_estimate.nationality_estimate import WikiNationalityEstimate

import os
os.environ['NationalityDatasets'] = "./data/"
estimate = WikiNationalityEstimate()

In [ ]:
data = estimate.get_raw_data()

In [ ]:
data.sample(n=10)

In [ ]:
data.fillna('', inplace=True)

In [ ]:
data.country.value_counts()

In [ ]:
data.country.unique()

In [ ]:
data['nationality to country'] = data.country.apply(lambda x: nationality_to_country(x))

In [ ]:
data[data['nationality to country']=='']

In [ ]:
data['country'] = data.apply(lambda x: x['nationality to country'] if x['nationality to country'] != '' else x.country, axis=1)

In [ ]:
changing_some_country_names = {
    'England': 'United Kingdom',
    'Scottish': 'United Kingdom',
    'Welsh': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'Northern Irish': 'Ireland',
    'Wales': 'United Kingdom',
    'Saudi': 'Saudi Arabia',
    'Filipina': 'Philippines',
    'Philippine': 'Philippines',
    'Burmese': 'Myanmar',
    'America': 'United States',
    'Abkhazia': 'Georgia',
    'South Ossetia': 'Georgia',
}
data.country = data.country.apply(lambda x: changing_some_country_names[x] if x in changing_some_country_names else x)

In [ ]:
data['standard'] = data.country.apply(lambda x: country_name_standardizer(x))

In [ ]:
data[(data.standard=='')&(data.country!='')].country.value_counts()

In [ ]:
data['country'] = data['standard']

In [ ]:
print(data.shape)
data = data[data.country != '']
print(data.shape)

In [ ]:
data.country.value_counts()

### Duplicates

In [ ]:
data[data.duplicated(subset=['name', 'country'], keep=False)]

In [ ]:
print(data.shape)
data.drop_duplicates(subset=['name', 'country'], keep='first', inplace=True)
print(data.shape)

#### Weird Names

In [ ]:
data.name.value_counts()[data.name.value_counts()>=2]

In [ ]:
data = data[(data.name!='He') & (data.name!='She' )]

In [ ]:
data.length = data.name.apply(lambda x: len(x))

In [ ]:
data[data.length>100]

In [ ]:
data['id'] = data.apply(lambda x:'wiki-estimate-'+str(x.name), axis=1)

In [ ]:
data[data['id'] == 'wiki-estimate-238627'].name.values[0]

### Saving the data

In [ ]:
data['full-name'] = data.name

In [ ]:
data['id'] = data.apply(lambda x:'wiki-estimate-'+str(x.name), axis=1)


In [ ]:
processed_data = data[['id', 'full-name', 'country']]
processed_data.set_index('id', inplace=True)

In [ ]:
estimate.data = processed_data

In [ ]:
estimate.save_data(processed_data)

In [ ]:
estimate.save_split_data()